# Neuro-Fuzzy + BERT Sentiment Classification

Trains a three-class sentiment classifier over employee review text, fusing a BERT
`[CLS]` encoding with fuzzy membership features derived from the text.

**This notebook is reference material, not a pipeline stage.** The sentiment scores the
recommendation engine consumes are produced by a separate subsystem and delivered to us
as a finished dataset; `run.py sentiment` reads that dataset directly. Nothing here is
imported or executed by `src/`.

It is kept, and its defects fixed, because the modelling choices are reported in the
paper and a reader may want to run them.

**Two defects were measured and corrected** (see `PROJECT.md` Section 0.8):

1. The rating-derived memberships `r_low / r_mid / r_high` reconstructed the label at
   **100.0% validation accuracy with the text ignored entirely** — BERT contributed
   nothing. No feature may be derived from `Overall_rating`; it builds the label only.
2. Rows with a null `Overall_rating` were defaulted to POSITIVE. All 14,330 of them are
   now dropped explicitly.

Also fixed: the TF-IDF vectoriser, SVD and scaler were fitted on train and validation
concatenated, and `vec`/`svd` were never persisted, so inference substituted zeros for
the three topic features the model was trained on.


In [ ]:
# Dependencies. On Colab, run this cell first; locally, vaderSentiment and
# scikit-fuzzy are already installed and only `transformers` is missing.
# %pip install transformers scikit-fuzzy vaderSentiment scikit-learn pandas tqdm --quiet
#
# Use a GPU runtime. Fine-tuning bert-base over ~65k reviews on CPU takes hours;
# on a T4 it is roughly 20-30 minutes at EPOCHS=4, BATCH_SIZE=32.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm

# Paths -- resolved relative to the project root so the notebook runs from anywhere.
PROJECT_ROOT = os.path.dirname(os.path.abspath('NeuroFuzzy_BERT_Sentiment.ipynb'))
DATA_CSV  = os.path.join(PROJECT_ROOT, 'employee_reviews.csv')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'neurofuzzy_bert_saved')
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1) Data load & text cleaning

`Overall_rating` is used to build the label and is then dropped from the feature path.
Null ratings are removed, not imputed: there were 14,330 of them and the previous
`map_rating` sent every one to POSITIVE.

In [ ]:
# Robust loader and cleaner
df = pd.read_csv(DATA_CSV)
print('rows loaded:', len(df))


def clean_text(s):
    if pd.isna(s):
        return ''
    s = str(s)
    s = re.sub(r'http\S+', '', s)
    s = re.sub(r'[^A-Za-z0-9\s.,!?"]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


# Text field: this corpus carries free text in Likes and Dislikes.
if 'Review_Text' in df.columns:
    df['text'] = df['Review_Text'].fillna('')
elif 'Likes' in df.columns or 'Dislikes' in df.columns:
    likes = df['Likes'].fillna('') if 'Likes' in df.columns else ''
    dislikes = df['Dislikes'].fillna('') if 'Dislikes' in df.columns else ''
    df['text'] = (likes + ' ' + dislikes).str.strip()
else:
    raise RuntimeError('No text-like column found in CSV')

df['text'] = df['text'].apply(clean_text)

# --- BUG-5 fix: drop null ratings explicitly. The old map_rating() returned 2
#     (POSITIVE) for every NaN, silently mislabelling 14,330 reviews.
if 'Overall_rating' not in df.columns:
    raise RuntimeError('Overall_rating column required to build labels')
n_before = len(df)
df = df[df['Overall_rating'].notna()].copy()
print('dropped for null Overall_rating:', n_before - len(df))


def map_rating(v):
    v = float(v)
    if v <= 2:
        return 0      # negative
    elif v == 3:
        return 1      # neutral
    return 2          # positive


df['label'] = df['Overall_rating'].astype(float).apply(map_rating)

# Drop empty texts -- a review with neither Likes nor Dislikes carries no signal.
n_before = len(df)
df = df[df['text'].str.strip() != ''].reset_index(drop=True)
print('dropped for empty text:', n_before - len(df))
print('examples remaining:', len(df))
print(df['label'].value_counts().sort_index().to_string())

## 2) Fuzzy feature extraction (text-derived only)

VADER runs on the review text and its compound polarity is fuzzified into
`v_neg / v_neu / v_pos`. The rating memberships `r_low / r_mid / r_high` that the earlier
revision used are **gone**: they were a copy of the label.

In [ ]:
# VADER polarity over the TEXT, fuzzified. No rating-derived feature appears here.
analyzer = SentimentIntensityAnalyzer()


def vader_compound(text):
    return analyzer.polarity_scores(text)['compound']


tqdm.pandas(desc='VADER')
df['vader_compound'] = df['text'].progress_apply(vader_compound)


def fuzzy_degrees(vader):
    """Map VADER compound in [-1, 1] to three gaussian membership degrees."""
    v = (vader + 1.0) / 2.0
    neg = fuzz.gaussmf(np.array([v]), 0.0, 0.15)[0]
    neu = fuzz.gaussmf(np.array([v]), 0.5, 0.15)[0]
    pos = fuzz.gaussmf(np.array([v]), 1.0, 0.15)[0]
    return [neg, neu, pos]


fuzzy_feats = np.vstack([fuzzy_degrees(v) for v in df['vader_compound'].values])
print('fuzzy feature shape:', fuzzy_feats.shape)

# Raw memberships, unscaled. Scaling is fitted on the training split only (next cell).
for i, name in enumerate(['v_neg', 'v_neu', 'v_pos']):
    df[name] = fuzzy_feats[:, i]

assert not any(c.startswith('r_') for c in df.columns), 'rating-derived feature leaked back in'
df[['text', 'label', 'v_neg', 'v_neu', 'v_pos']].head()

## 3) Tokenization and PyTorch Dataset

The split happens **first**. The TF-IDF vectoriser, the SVD and the feature scaler are
all fitted on the training split alone and then applied to validation --- the previous
revision fitted them on train and validation concatenated.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Split FIRST, then fit every transform on train only.
train_idx, val_idx = train_test_split(
    df.index, test_size=0.1, random_state=SEED, stratify=df['label'])
train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)

# Surface features
for d in (train_df, val_df):
    d['word_count'] = d['text'].apply(lambda t: 0 if not t else len(t.split()))
    d['avg_word_len'] = d['text'].apply(
        lambda t: 0.0 if not t else sum(len(w) for w in t.split()) / len(t.split()))

# Topic components -- fitted on TRAIN ONLY, then applied to val.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

TOPIC_DIM = 3
vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
train_tfidf = vec.fit_transform(train_df['text'].tolist())
svd = TruncatedSVD(n_components=TOPIC_DIM, random_state=SEED)
train_topic = svd.fit_transform(train_tfidf)
val_topic = svd.transform(vec.transform(val_df['text'].tolist()))

for i in range(TOPIC_DIM):
    train_df[f'topic_{i}'] = train_topic[:, i]
    val_df[f'topic_{i}'] = val_topic[:, i]


def tokenize_texts(texts, batch_size=512):
    all_enc = {'input_ids': [], 'attention_mask': []}
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i + batch_size], truncation=True,
                          padding='max_length', max_length=128, return_tensors='pt')
        all_enc['input_ids'].append(batch['input_ids'])
        all_enc['attention_mask'].append(batch['attention_mask'])
    all_enc['input_ids'] = torch.cat(all_enc['input_ids'], dim=0)
    all_enc['attention_mask'] = torch.cat(all_enc['attention_mask'], dim=0)
    return all_enc


train_enc = tokenize_texts(train_df['text'])
val_enc = tokenize_texts(val_df['text'])

# NOTE: no r_low / r_mid / r_high. Every column below is derived from the text.
fuzzy_cols = (['v_neg', 'v_neu', 'v_pos']
              + ['word_count', 'avg_word_len']
              + [f'topic_{i}' for i in range(TOPIC_DIM)])

from sklearn.preprocessing import StandardScaler
fuzzy_scaler = StandardScaler().fit(train_df[fuzzy_cols].values)
train_fuzzy = torch.tensor(fuzzy_scaler.transform(train_df[fuzzy_cols].values), dtype=torch.float32)
val_fuzzy = torch.tensor(fuzzy_scaler.transform(val_df[fuzzy_cols].values), dtype=torch.float32)
train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)
val_labels = torch.tensor(val_df['label'].values, dtype=torch.long)


class NeuroFuzzyDataset(Dataset):
    def __init__(self, enc, fuzzy, labels):
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.fuzzy = fuzzy
        self.labels = labels

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'fuzzy': self.fuzzy[idx],
            'labels': self.labels[idx],
        }


train_dataset = NeuroFuzzyDataset(train_enc, train_fuzzy, train_labels)
val_dataset = NeuroFuzzyDataset(val_enc, val_fuzzy, val_labels)
print('train size', len(train_dataset), 'val size', len(val_dataset),
      'fuzzy dim', train_fuzzy.size(1))

In [ ]:
# Configuration / hyperparameters
# EPOCHS was 100, which with PATIENCE=2 was dead code and on CPU is days of compute.
EPOCHS = 4
BATCH_SIZE = 32
LR = 2e-5
PATIENCE = 2
MONITOR_METRIC = 'val_f1'   # 'val_loss' or 'val_f1'
FREEZE_INITIAL = True       # start with BERT frozen
UNFREEZE_EPOCH = 2          # 1-based epoch at which to unfreeze; None to never unfreeze

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('device:', device)
if device.type == 'cpu':
    print('WARNING: CPU training over ~65k reviews will take hours. Prefer a GPU runtime.')

## 4) Model: BERT encoder + fuzzy features classifier

In [ ]:
import torch.nn as nn


class NeuroFuzzyBert(nn.Module):
    def __init__(self, bert_name='bert-base-uncased', fuzzy_dim=8, num_labels=3, freeze_bert=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden + fuzzy_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, fuzzy):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]      # [CLS]
        x = torch.cat([pooled, fuzzy], dim=1)
        x = self.dropout(x)
        return self.classifier(x)


model = NeuroFuzzyBert(bert_name='bert-base-uncased',
                       fuzzy_dim=train_fuzzy.size(1),
                       num_labels=int(df['label'].nunique()),
                       freeze_bert=FREEZE_INITIAL)
model.to(device)
print('fuzzy_dim', train_fuzzy.size(1), '| num_labels', int(df['label'].nunique()))

## 5) Training loop with Early Stopping and model saving

In [ ]:
from transformers import get_linear_schedule_with_warmup
import json

def _get_optimizer_scheduler(model, lr, train_loader, epochs_left):
    opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    total_steps = len(train_loader) * max(1, epochs_left)
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
    return opt, sched

def train_model(model, train_dataset, val_dataset, device, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE, monitor=MONITOR_METRIC, unfreeze_epoch=UNFREEZE_EPOCH):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    optimizer, scheduler = _get_optimizer_scheduler(model, lr, train_loader, epochs)
    loss_fn = nn.CrossEntropyLoss()
    # best metric depends on monitor: lower better for loss, higher better for f1
    if monitor == 'val_loss':
        best_metric = float('inf')
        better = lambda cur, best: cur < best - 1e-4
    else:
        best_metric = -float('inf')
        better = lambda cur, best: cur > best + 1e-4

    best_epoch = 0
    patience_counter = 0
    best_state = None

    for epoch in range(1, epochs+1):
        # optionally unfreeze BERT at a given epoch (1-based)
        if unfreeze_epoch is not None and epoch == unfreeze_epoch:
            print('Unfreezing BERT encoder at epoch', epoch)
            for p in model.bert.parameters():
                p.requires_grad = True
            # rebuild optimizer & scheduler to include newly-unfrozen params for remaining epochs
            remaining_epochs = epochs - epoch + 1
            optimizer, scheduler = _get_optimizer_scheduler(model, lr, train_loader, remaining_epochs)

        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f'Train Epoch {epoch}'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            fuzzy = batch['fuzzy'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask, fuzzy=fuzzy)
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_losses.append(loss.item())
        avg_train_loss = np.mean(train_losses) if len(train_losses) > 0 else 0.0

        # validation
        model.eval()
        val_losses = []
        preds = []
        trues = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Validation'):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                fuzzy = batch['fuzzy'].to(device)
                labels = batch['labels'].to(device)
                logits = model(input_ids=input_ids, attention_mask=attention_mask, fuzzy=fuzzy)
                loss = loss_fn(logits, labels)
                val_losses.append(loss.item())
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
                trues.extend(labels.cpu().numpy().tolist())
        avg_val_loss = np.mean(val_losses) if len(val_losses) > 0 else 0.0
        val_acc = accuracy_score(trues, preds) if len(trues) > 0 else 0.0
        val_f1 = f1_score(trues, preds, average='macro') if len(trues) > 0 else 0.0
        print(f'Epoch {epoch}: train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}')

        # choose metric for early stopping
        cur_metric = avg_val_loss if monitor == 'val_loss' else val_f1
        if better(cur_metric, best_metric):
            best_metric = cur_metric
            best_epoch = epoch
            patience_counter = 0
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            # save best model checkpoint and tokenizer for deployment
            torch.save(best_state, os.path.join(MODEL_DIR, 'best_model.pt'))
            try:
                tokenizer.save_pretrained(MODEL_DIR)
            except Exception as e:
                print('Tokenizer save failed:', e)
            # also save metadata about fuzzy cols and label mapping (force Python ints for JSON)
            label_values = [int(x) for x in df['label'].unique().tolist()]
            meta = {'fuzzy_cols': fuzzy_cols, 'monitor': monitor, 'labels': label_values}
            with open(os.path.join(MODEL_DIR, 'metadata.json'), 'w') as f:
                json.dump(meta, f, indent=2)
            print('Saved best model at epoch', epoch)
        else:
            patience_counter += 1
            print('No improvement, patience', patience_counter, '/', patience)
            if patience_counter >= patience:
                print('Early stopping triggered. Restoring best model from epoch', best_epoch)
                # restore best state
                if best_state is not None:
                    model.load_state_dict(best_state)
                    torch.save(best_state, os.path.join(MODEL_DIR, 'best_model_final.pt'))
                    try:
                        tokenizer.save_pretrained(MODEL_DIR)
                    except Exception:
                        pass
                return model
    # finished epochs
    # final save
    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(best_state, os.path.join(MODEL_DIR, 'best_model_final.pt'))
        try:
            tokenizer.save_pretrained(MODEL_DIR)
        except Exception:
            pass
    return model

# Run training with early stopping using the top-level config variables
trained = train_model(model, train_dataset, val_dataset, device, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE, monitor=MONITOR_METRIC, unfreeze_epoch=UNFREEZE_EPOCH)


Train Epoch 1:   0%|          | 0/3670 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6) Load best model and evaluate on validation set

In [ ]:
# Load best checkpoint and run full eval
best_path = os.path.join(MODEL_DIR, 'best_model_final.pt')
if os.path.exists(best_path):
    state = torch.load(best_path, map_location=device)
    model.load_state_dict(state)

# Eval on val set
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
model.eval()
preds = []
trues = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        fuzzy = batch['fuzzy'].to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask, fuzzy=fuzzy)
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        trues.extend(labels.cpu().numpy().tolist())

print('Val accuracy:', accuracy_score(trues, preds))
print(classification_report(trues, preds))


In [ ]:
# --- Save a bundle that is sufficient to reproduce the feature pipeline ---
import pickle

bundle_path = os.path.join(MODEL_DIR, 'model_bundle.pkl')
model_state_for_bundle = best_state if 'best_state' in globals() and best_state is not None else None
if model_state_for_bundle is None:
    ckpt_path = os.path.join(MODEL_DIR, 'best_model_final.pt')
    if os.path.exists(ckpt_path):
        model_state_for_bundle = torch.load(ckpt_path, map_location='cpu')

# The earlier revision omitted `vec` and `svd`, so inference substituted zeros for the
# three topic features the model was trained on. Persist the whole transform chain.
bundle = {
    'model_state': model_state_for_bundle,
    'fuzzy_scaler': fuzzy_scaler,
    'fuzzy_cols': fuzzy_cols,
    'tfidf_vectorizer': vec,
    'svd': svd,
    'topic_dim': TOPIC_DIM,
    'labels': sorted(df['label'].unique()),
    'max_length': 128,
    'bert_name': 'bert-base-uncased',
}
with open(bundle_path, 'wb') as f:
    pickle.dump(bundle, f)
tokenizer.save_pretrained(MODEL_DIR)
print('Saved bundle to', bundle_path)
print('Saved tokenizer to', MODEL_DIR)

In [ ]:
# --- Rebuild the feature pipeline from the bundle and verify it matches training ---
import pickle

with open(os.path.join(MODEL_DIR, 'model_bundle.pkl'), 'rb') as f:
    bundle = pickle.load(f)
print('Bundle keys:', list(bundle.keys()))

model_loaded = NeuroFuzzyBert(bert_name=bundle['bert_name'],
                              fuzzy_dim=len(bundle['fuzzy_cols']),
                              num_labels=len(bundle['labels']))
if bundle.get('model_state') is not None:
    model_loaded.load_state_dict(bundle['model_state'])
model_loaded.to(device).eval()
tokenizer_loaded = AutoTokenizer.from_pretrained(MODEL_DIR)


def build_features(texts):
    """Training-identical feature construction. Uses the persisted vectoriser and SVD
    rather than the zero-filled placeholders the earlier revision used."""
    texts = [clean_text(t) for t in texts]
    rows = []
    topic = bundle['svd'].transform(bundle['tfidf_vectorizer'].transform(texts))
    for i, t in enumerate(texts):
        v = analyzer.polarity_scores(t)['compound']
        wc = 0 if not t else len(t.split())
        awl = 0.0 if not t else sum(len(w) for w in t.split()) / len(t.split())
        rows.append(fuzzy_degrees(v) + [wc, awl] + list(topic[i]))
    return bundle['fuzzy_scaler'].transform(np.array(rows, dtype=float))


# Verify: features rebuilt from raw text must equal the training-time features.
_check = build_features(val_df['text'].head(64).tolist())
_train_time = fuzzy_scaler.transform(val_df[fuzzy_cols].head(64).values)
print('max abs deviation vs training-time features:', np.abs(_check - _train_time).max())
assert np.allclose(_check, _train_time, atol=1e-6), 'inference features diverge from training'
print('OK -- inference pipeline reproduces training features exactly.')

## Notes and further improvements
- You can tune patience, metric to monitor (val_f1), batch size, and learning rate.
- Consider freezing BERT initially then unfreezing.
- Fuzzy feature design can be extended with more signals (word-counts, topic scores).
- Save and load tokenizer/model directories for deployment.